# Elemental Gallium Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MarsZDF/gallium/blob/main/demo.ipynb)

**Image gallery grids, experiment tracking, and A/B comparison for image generation workflows.**

This notebook demonstrates the core features of `elemental-gallium`:
- Experiment tracking with SQLite
- Querying and filtering experiments
- Building comparison grids
- A/B comparison
- Export to CSV, JSON, and HTML

## Installation

In [ ]:
# Install gallium with grid support
!pip install -q elemental-gallium[grid]

## Setup

For this demo, we'll create some sample images to work with.

In [ ]:
import gallium
from PIL import Image
import os

# Create output directory
os.makedirs("demo_outputs", exist_ok=True)

# Initialize gallium with a fresh database
gallium.init("demo.db")

print(f"Gallium version: {gallium.__version__}")

In [ ]:
# Create sample images (simulating generated images)
def create_sample_image(color: str, size: tuple = (256, 256)) -> Image.Image:
    """Create a simple colored image for demo purposes."""
    return Image.new("RGB", size, color)

# Generate sample "experiments"
colors = ["#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#FFEAA7", "#DDA0DD"]
prompts = [
    "cyberpunk city at night",
    "cyberpunk city at night",
    "forest in morning mist",
    "forest in morning mist",
    "portrait of a robot",
    "portrait of a robot",
]
seeds = [42, 123, 42, 123, 42, 123]
models = ["flux.2-pro", "flux.2-pro", "flux.2-max", "flux.2-max", "flux.2-flex", "flux.2-flex"]

for i, (color, prompt, seed, model) in enumerate(zip(colors, prompts, seeds, models)):
    img = create_sample_image(color)
    path = f"demo_outputs/sample_{i}.png"
    img.save(path)
    
    gallium.log(
        prompt=prompt,
        seed=seed,
        path=path,
        model=model,
        width=256,
        height=256,
        duration_ms=1000 + i * 100,
        params={"guidance": 7.5, "steps": 28}
    )

print(f"Created {len(colors)} sample experiments")

## 1. Querying Experiments

Use `find()` to query experiments with flexible filters.

In [ ]:
# Find all experiments
all_experiments = gallium.find()
print(f"Total experiments: {len(all_experiments)}")

# Show the first one
exp = all_experiments[0]
print(f"\nFirst experiment:")
print(f"  ID: {exp.id}")
print(f"  Prompt: {exp.prompt}")
print(f"  Seed: {exp.seed}")
print(f"  Model: {exp.model}")
print(f"  Duration: {exp.duration_ms}ms")

In [ ]:
# Filter by prompt content
cyberpunk = gallium.find(prompt__contains="cyberpunk")
print(f"Cyberpunk experiments: {len(cyberpunk)}")

# Filter by seed
seed_42 = gallium.find(seed=42)
print(f"Seed 42 experiments: {len(seed_42)}")

# Filter by model
flux_pro = gallium.find(model="flux.2-pro")
print(f"FLUX.2 Pro experiments: {len(flux_pro)}")

In [ ]:
# Get recent experiments
recent = gallium.recent(3)
print("3 most recent experiments:")
for exp in recent:
    print(f"  [{exp.id}] {exp.prompt[:30]}... (seed={exp.seed})")

## 2. Building Comparison Grids

Create visual grids to compare experiments side-by-side.

In [ ]:
# Basic grid from experiments
all_exp = gallium.find()
grid_img = gallium.grid(all_exp, cols=3)
grid_img.save("demo_outputs/basic_grid.png")
display(grid_img)

In [ ]:
# Grid with labels
cyberpunk = gallium.find(prompt__contains="cyberpunk")
labeled_grid = gallium.grid(
    cyberpunk,
    cols=2,
    labels=[f"seed={e.seed}" for e in cyberpunk],
    padding=15,
    background="#1a1a2e"
)
labeled_grid.save("demo_outputs/labeled_grid.png")
display(labeled_grid)

## 3. Matrix Grid

Compare experiments across two dimensions (e.g., model vs. seed).

In [ ]:
# Create a matrix comparing models vs seeds
all_exp = gallium.find()
matrix = gallium.matrix_grid(
    all_exp,
    rows="model",
    cols="seed",
    max_size=128,
    show_labels=True
)
matrix.save("demo_outputs/matrix_grid.png")
display(matrix)

## 4. A/B Comparison

Compare two specific images side-by-side.

In [ ]:
# Compare two experiments
cyberpunk = gallium.find(prompt__contains="cyberpunk")
if len(cyberpunk) >= 2:
    result = gallium.compare(
        cyberpunk[0].path,
        cyberpunk[1].path,
        labels=(f"seed={cyberpunk[0].seed}", f"seed={cyberpunk[1].seed}")
    )
    comparison = result.grid()
    comparison.save("demo_outputs/comparison.png")
    display(comparison)

## 5. Star and Annotate

Mark your best experiments and add notes.

In [ ]:
# Star the best experiments
best = gallium.find(prompt__contains="cyberpunk", seed=42)
if best:
    gallium.star(best[0].id)
    gallium.annotate(best[0].id, "Best cyberpunk composition")
    print(f"Starred experiment {best[0].id}")

# Find all starred
starred = gallium.find(starred=True)
print(f"\nStarred experiments: {len(starred)}")
for exp in starred:
    print(f"  [{exp.id}] {exp.prompt[:30]}... - {exp.notes}")

## 6. Export Data

Export your experiments to CSV, JSON, or HTML.

In [ ]:
# Export to CSV
gallium.export("csv", path="demo_outputs/experiments.csv")
print("Exported to CSV")

# Export to JSON
gallium.export("json", path="demo_outputs/experiments.json")
print("Exported to JSON")

# Export to HTML gallery
gallium.export("html", path="demo_outputs/gallery.html", title="Demo Gallery")
print("Exported to HTML gallery")

In [ ]:
# View the JSON export
import json

with open("demo_outputs/experiments.json") as f:
    data = json.load(f)
    
print(f"Exported {len(data)} experiments")
print("\nFirst experiment:")
print(json.dumps(data[0], indent=2))

## 7. Cleanup

In [ ]:
# Optional: Clean up demo files
import shutil

# Uncomment to delete demo files
# shutil.rmtree("demo_outputs", ignore_errors=True)
# os.remove("demo.db") if os.path.exists("demo.db") else None
# print("Cleaned up demo files")

## Next Steps

- Check out the [FLUX.2 Demo](flux2_demo.ipynb) for live image generation with BFL API
- See the [examples/](https://github.com/MarsZDF/gallium/tree/main/examples) directory for API integration examples
- Read the full [README](https://github.com/MarsZDF/gallium) for API reference